# Malware Detection with SGD
LSTM classifier trained with Stochastic Gradient Descent on opcode sequences.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Config

In [ ]:
MAX_SEQ_LEN = 200
VOCAB_SIZE = 2000
BATCH_SIZE = 64
NUM_EPOCHS = 30
LR = 0.01

## Load Data
Swap the synthetic block for your real CSV when ready.

In [ ]:
# ---- real data ----
df = pd.read_csv('malware_opcodes.csv')
print(f'{len(df)} samples, {df["label"].nunique()} classes')

## Preprocess
Build vocab, encode sequences to integers, pad/truncate to fixed length.

In [ ]:
# build vocab from top N opcodes (0 = PAD, 1 = UNK)
all_tokens = ' '.join(df['opcodes']).split()
most_common = [tok for tok, _ in Counter(all_tokens).most_common(VOCAB_SIZE - 2)]
token2idx = {tok: i+2 for i, tok in enumerate(most_common)}

def encode_and_pad(seq_str):
    ids = [token2idx.get(t, 1) for t in seq_str.split()]
    ids = ids[:MAX_SEQ_LEN]
    ids += [0] * (MAX_SEQ_LEN - len(ids))
    return ids

X = torch.tensor([encode_and_pad(s) for s in df['opcodes']], dtype=torch.long)
y = torch.tensor(df['label'].values, dtype=torch.long)
num_classes = y.unique().size(0)
vocab_size = len(token2idx) + 2

print(f'Vocab: {vocab_size} tokens, Sequence shape: {X.shape}')

In [ ]:
# split and wrap in DataLoaders
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## Model
Embedding → LSTM → Linear. All built-in PyTorch layers.

In [ ]:
class MalwareLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

model = MalwareLSTM().to(device)
print(model)

## SGD Optimizer and Loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9)

#add rr in cell below

## Training Loop

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- train ----
    model.train()
    total_loss, correct, total = 0, 0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)

    history['train_loss'].append(total_loss / total)
    history['train_acc'].append(correct / total)

    # ---- validate ----
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            val_loss += criterion(out, yb).item() * xb.size(0)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_total += yb.size(0)

    history['val_loss'].append(val_loss / val_total)
    history['val_acc'].append(val_correct / val_total)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:>2} | '
              f'Train Loss: {history["train_loss"][-1]:.4f} Acc: {history["train_acc"][-1]:.2%} | '
              f'Val Loss: {history["val_loss"][-1]:.4f} Acc: {history["val_acc"][-1]:.2%}')

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss (SGD)')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy (SGD)')
ax2.legend()

plt.tight_layout()
plt.show()

## Test Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, str(cm[i,j]), ha='center', va='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (SGD)')
plt.colorbar()
plt.tight_layout()
plt.show()

## Save for RR Comparison

In [ ]:
import json

with open('sgd_history.json', 'w') as f:
    json.dump(history, f)

torch.save(model.state_dict(), 'sgd_model.pt')
print('Saved sgd_history.json and sgd_model.pt')